In [1]:
# -*- coding: utf-8 -*-
"""
Created on 2026-06-30

@author:       Oscar Trevizo
@institution:  Harvard Extension School
@credential:   Graduate Certificate in Data Science (2023)
@context:      Independent project — applying graduate-level concepts to real-world data
@environment:  Python 3.14.3 | myenv | MacBook Air M5

Agentic Semantic Layer Necessity
==================================

Hypothesis:
    LLM agents reason more accurately and consistently when querying
    data through semantic business names than when querying raw
    technical column names directly.

Experiment:
    Same question asked to three models under two conditions:

    Condition A — With semantic layer:
        Agents call query_product(business_name='net_migration_rate').
        Business names are governed, human-readable, and source-agnostic.

    Condition B — Without semantic layer:
        Agents call query_raw(column_name='NetMigrationRate_per_Kpop').
        Agents must discover and use raw technical column names via
        list_columns(). No semantic mediation.

    Question: "Which country in the top 20 GDP has the highest
               migration rate?"

    Models: GPT-4o-mini · Haiku · Sonnet (same as Experiment 1)

Neurosymbolic framing:
    Condition A tests the full hybrid system — symbolic naming + neural
    reasoning. Condition B removes the symbolic naming layer, leaving
    neural reasoning to operate on raw technical names alone.
    Comparing the two isolates the contribution of semantic governance
    to reasoning quality.

Precursor: machine_learning/agentic_model_reasoning_divergence.ipynb
Support   : ../python_vignettes/data_products/data_product_lib.py

Revision History:
    2026-06-30  Original development
                - Two-condition design: with vs without semantic layer
                - query_raw() + list_columns() tools for Condition B
                - Shared run_agent() parameterised by tool set
"""

'\nCreated on 2026-06-30\n\n@author:       Oscar Trevizo\n@institution:  Harvard Extension School\n@credential:   Graduate Certificate in Data Science (2023)\n@context:      Independent project — applying graduate-level concepts to real-world data\n@environment:  Python 3.14.3 | myenv | MacBook Air M5\n\nAgentic Semantic Layer Necessity\n==================================\n\nHypothesis:\n    LLM agents reason more accurately and consistently when querying\n    data through semantic business names than when querying raw\n    technical column names directly.\n\nExperiment:\n    Same question asked to three models under two conditions:\n\n    Condition A — With semantic layer:\n        Agents call query_product(business_name=\'net_migration_rate\').\n        Business names are governed, human-readable, and source-agnostic.\n\n    Condition B — Without semantic layer:\n        Agents call query_raw(column_name=\'NetMigrationRate_per_Kpop\').\n        Agents must discover and use raw techni

## Hypothesis

> **LLM agents reason more accurately and consistently when querying data
> through semantic business names than when querying raw technical column names.**

### Two conditions

| | Condition A | Condition B |
|---|---|---|
| **Query tool** | `query_product(business_name=...)` | `query_raw(column_name=...)` |
| **Discovery tool** | semantic names from product card | `list_columns()` — raw DataFrame columns |
| **What agent sees** | `'net_migration_rate'`, `'gdp'` | `'NetMigrationRate_per_Kpop'`, `'GDP_USD'` |
| **Semantic layer** | Present | Absent |

Everything else is identical: same data, same pipeline, same merge, same models.

### What the semantic layer provides

- **Abstraction:** `'net_migration_rate'` hides the raw column name
- **Stability:** business names survive schema changes in the raw data
- **Discoverability:** agents can ask "what metrics exist?" in human terms
- **Disambiguation:** unit and description are registered alongside the name

The question is whether removing this layer degrades agent reasoning —
and whether the degradation is consistent across models or model-dependent.

In [2]:
import io
import json
import os
import sys
from datetime import datetime, timezone

import anthropic
import numpy as np
import pandas as pd
from openai import OpenAI

sys.path.insert(0, '../python_vignettes/data_products/')
from data_product_lib import (
    DataProductMetadata,
    SemanticLayer,
    LineageTracker,
    DataProduct,
)

anthropic_client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
openai_client    = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

print(f"Anthropic SDK : {anthropic.__version__}")
import openai as _oai
print(f"OpenAI SDK    : {_oai.__version__}")

Anthropic SDK : 0.88.0
OpenAI SDK    : 2.30.0


## Experimental Design

| Element | Value |
|---|---|
| **Question** | *"Which country in the top 20 GDP has the highest migration rate?"* |
| **Models** | GPT-4o-mini · Haiku · Sonnet |
| **Conditions** | A: with semantic layer · B: without semantic layer |
| **Runs** | 6 total (3 models × 2 conditions) |
| **Controlled** | Data, pipeline, merge, query order, registry |
| **Variable** | Presence or absence of the semantic layer |

### What we measure

- **Correctness** — does the model identify Canada (+11.039, 2023)?
- **Year filter** — does the model anchor to 2023?
- **Column selection** — in Condition B, does the model choose the right raw column?
- **Tool call count** — does the absence of semantic names add or remove calls?
- **Variance** — do models diverge more in Condition B than Condition A?

In [3]:
UN_FILE            = '../data/WPP2024_GEN_F01_DEMOGRAPHIC_INDICATORS_COMPACT_20260327.xlsx'
WB_GDP_FILE        = '../data/WB_GDP.xls'
WB_GDP_GROWTH_FILE = '../data/WB_GDP_growth.xls'


def load_un_wpp(filepath, year_start=1961, year_end=2023):
    df = pd.read_excel(
        filepath, sheet_name='Estimates', skiprows=16, thousands=' ',
        usecols=[
            'Region, subregion, country or area *', 'ISO3 Alpha-code',
            'ISO2 Alpha-code', 'Type', 'Year',
            'Total Population, as of 1 July (thousands)',
            'Median Age, as of 1 July (years)',
            'Population Growth Rate (percentage)',
            'Total Fertility Rate (live births per woman)',
            'Life Expectancy at Birth, both sexes (years)',
            'Net Number of Migrants (thousands)',
            'Net Migration Rate (per 1,000 population)',
        ]
    )
    df = df.rename(columns={
        'Region, subregion, country or area *'         : 'Location',
        'ISO3 Alpha-code'                              : 'ISO3',
        'ISO2 Alpha-code'                              : 'ISO2',
        'Type'                                         : 'LocType',
        'Total Population, as of 1 July (thousands)'  : 'Population_Ks',
        'Median Age, as of 1 July (years)'             : 'MedAge',
        'Population Growth Rate (percentage)'          : 'PopulationGrowthRate',
        'Total Fertility Rate (live births per woman)' : 'FertilityRate_births_per_woman',
        'Life Expectancy at Birth, both sexes (years)' : 'LifeExpectancy',
        'Net Number of Migrants (thousands)'           : 'NetMigrants_Ks',
        'Net Migration Rate (per 1,000 population)'    : 'NetMigrationRate_per_Kpop',
    })
    df = df.dropna(subset=['Year'])
    df['Year'] = df['Year'].astype(np.int64)
    df = df[(df['Year'] >= year_start) & (df['Year'] <= year_end)]
    df.loc[df['LocType'] == 'Country/Area', 'LocType'] = 'Country'
    df['ReceivesMigrants']    = (df['NetMigrants_Ks'] > 0).astype(int)
    df['ImmigrantsEmigrants'] = df['NetMigrants_Ks'].apply(
        lambda x: 'Immigrants' if x > 0 else 'Emigrants')
    return df.reset_index(drop=True)


def filter_countries(df):
    return df[df['LocType'] == 'Country'].reset_index(drop=True)


def clean_types(df):
    for col in ['Population_Ks', 'MedAge', 'PopulationGrowthRate',
                'FertilityRate_births_per_woman', 'LifeExpectancy',
                'NetMigrants_Ks', 'NetMigrationRate_per_Kpop']:
        df = df.copy()
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def load_wb_gdp(gdp_file, gdp_growth_file, year_start=1961, year_end=2023):
    def _load_one(filepath, indicator):
        df = pd.read_excel(filepath, skiprows=3, sheet_name='Data')
        df = df.rename(columns={'Country Code': 'ISO3'})
        df = df.drop(columns=['Country Name', 'Indicator Name', 'Indicator Code'])
        df = df.drop(columns=[c for c in df.columns if str(c) == '2025'], errors='ignore')
        df = df.set_index('ISO3').stack().reset_index()
        df.columns = ['ISO3', 'Year', indicator]
        df['Year']    = df['Year'].astype(np.int64)
        df[indicator] = pd.to_numeric(df[indicator], errors='coerce')
        return df
    df = pd.merge(_load_one(gdp_file, 'GDP_USD'),
                  _load_one(gdp_growth_file, 'GDP_growth_pct'),
                  on=['ISO3', 'Year'])
    df = df[(df['Year'] >= year_start) & (df['Year'] <= year_end)]
    return df.reset_index(drop=True)


def _merge_data_products(dp1, dp2, on):
    collisions = set(dp1.semantic_layer.to_dict()) & set(dp2.semantic_layer.to_dict())
    if collisions:
        raise ValueError(f'Semantic name collision(s): {sorted(collisions)}.')
    merged_df = pd.merge(dp1.data, dp2.data, on=on, how='inner')
    merged_sl = SemanticLayer()
    for name, e in dp1.semantic_layer.to_dict().items():
        merged_sl.register(name, e['column'], e['unit'],
                           e['description'], e['source_system'])
    for name, e in dp2.semantic_layer.to_dict().items():
        merged_sl.register(name, e['column'], e['unit'],
                           e['description'], e['source_system'])
    merged_lin = LineageTracker()
    for s in dp1.lineage.to_list():
        merged_lin.log(f"dp1/{s['step']}", s['operation'],
                       s['input_shape'], s['output_shape'], s['notes'])
    for s in dp2.lineage.to_list():
        merged_lin.log(f"dp2/{s['step']}", s['operation'],
                       s['input_shape'], s['output_shape'], s['notes'])
    merged_lin.log('merge', 'inner_join', dp1.data.shape, merged_df.shape,
                   f'keys: {"+".join(on)}')
    return merged_df, merged_sl, merged_lin


print("Pipeline functions ready.")

Pipeline functions ready.


In [4]:
# Shared across both conditions — only the query tool differs
PRODUCT_REGISTRY: dict = {}

SOURCE_CONFIGS = {
    "UN_WPP":         {"description": "UN World Population Prospects 2024"},
    "WORLD_BANK_GDP": {"description": "World Bank WDI — GDP indicators"},
}


def list_available_sources() -> dict:
    return {"available_sources": SOURCE_CONFIGS,
            "loaded": list(PRODUCT_REGISTRY.keys())}


def load_source(source_name: str,
                year_start: int = 1961, year_end: int = 2023) -> dict:
    if source_name in PRODUCT_REGISTRY:
        dp = PRODUCT_REGISTRY[source_name]
        return {"status": "already_loaded", "product_name": source_name,
                "shape": list(dp.data.shape)}
    if source_name == "UN_WPP":
        lin = LineageTracker()
        raw  = load_un_wpp(UN_FILE, year_start, year_end)
        lin.log("1-load",   "load_un_wpp",        (0, 0),    raw.shape,  f"{year_start}-{year_end}")
        ctry = filter_countries(raw)
        lin.log("2-filter", "filter_countries",    raw.shape, ctry.shape, "LocType==Country")
        df   = clean_types(ctry)
        lin.log("3-clean",  "clean_types",          ctry.shape, df.shape, "to_numeric")
        sl = SemanticLayer()
        sl.register("net_migration_rate", "NetMigrationRate_per_Kpop", "per 1,000 population",
                    "Net migrants per 1,000 residents", source_system="UN WPP 2024")
        sl.register("net_migrants", "NetMigrants_Ks", "thousands",
                    "Net number of migrants", source_system="UN WPP 2024")
        sl.register("population", "Population_Ks", "thousands",
                    "Total population, as of 1 July", source_system="UN WPP 2024")
        sl.register("fertility_rate", "FertilityRate_births_per_woman", "births per woman",
                    "Total fertility rate", source_system="UN WPP 2024")
        sl.register("life_expectancy", "LifeExpectancy", "years",
                    "Life expectancy at birth", source_system="UN WPP 2024")
        meta = DataProductMetadata(
            name="UN WPP Demographics", description="Country-year panel, UN WPP 2024",
            domain="Demographics", owner="Experiment",
            source="UN World Population Prospects 2024",
            source_url="https://population.un.org/wpp/",
            license="CC BY 3.0 IGO", version="1.0",
            refresh_frequency="Every 2 years",
            created_at=datetime.now(timezone.utc).isoformat())
        dp = DataProduct(df, meta, sl, lin)
    elif source_name == "WORLD_BANK_GDP":
        lin = LineageTracker()
        df  = load_wb_gdp(WB_GDP_FILE, WB_GDP_GROWTH_FILE, year_start, year_end)
        lin.log("1-load", "load_wb_gdp", (0, 0), df.shape, f"{year_start}-{year_end}")
        sl = SemanticLayer()
        sl.register("gdp", "GDP_USD", "constant USD",
                    "GDP (NY.GDP.MKTP.KD)", source_system="World Bank WDI")
        sl.register("gdp_growth", "GDP_growth_pct", "% per year",
                    "GDP growth (NY.GDP.MKTP.KD.ZG)", source_system="World Bank WDI")
        meta = DataProductMetadata(
            name="World Bank GDP", description="Country-year panel, World Bank WDI",
            domain="Economics", owner="Experiment",
            source="World Bank World Development Indicators",
            source_url="https://databank.worldbank.org/",
            license="CC BY 4.0", version="1.0",
            refresh_frequency="Annual",
            created_at=datetime.now(timezone.utc).isoformat())
        dp = DataProduct(df, meta, sl, lin)
    else:
        return {"error": f"Unknown source '{source_name}'."}
    PRODUCT_REGISTRY[source_name] = dp
    return {"status": "loaded", "product_name": source_name,
            "shape": list(dp.data.shape),
            "semantic_names": list(dp.semantic_layer.to_dict().keys())}


def merge_sources(source1_name, source2_name, product_name="MERGED"):
    missing = [n for n in (source1_name, source2_name) if n not in PRODUCT_REGISTRY]
    if missing:
        return {"error": f"Not in registry: {missing}"}
    dp1, dp2 = PRODUCT_REGISTRY[source1_name], PRODUCT_REGISTRY[source2_name]
    try:
        mdf, msl, mlin = _merge_data_products(dp1, dp2, on=["ISO3", "Year"])
    except ValueError as exc:
        return {"error": str(exc)}
    meta = DataProductMetadata(
        name=product_name, description=f"Merged: {source1_name}+{source2_name}",
        domain="Multi-domain", owner="Experiment",
        source=f"{dp1.metadata.source}+{dp2.metadata.source}",
        source_url="", license="", version="1.0",
        refresh_frequency="On demand",
        created_at=datetime.now(timezone.utc).isoformat())
    dp = DataProduct(mdf, meta, msl, mlin)
    PRODUCT_REGISTRY[product_name] = dp
    return {"status": "merged", "product_name": product_name,
            "shape": list(dp.data.shape),
            "semantic_names": sorted(dp.semantic_layer.to_dict().keys())}


print("Shared tools ready (list_available_sources, load_source, merge_sources).")

Shared tools ready (list_available_sources, load_source, merge_sources).


---
## Condition A — With Semantic Layer

Agents use `query_product(business_name=...)` to retrieve data.
Business names are human-readable and governed by the semantic layer.

In [5]:
# ── Condition A: semantic query tool ────────────────────────────────────────

def query_product(product_name, business_name, year=None, top_n=10):
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    dp  = PRODUCT_REGISTRY[product_name]
    sl  = dp.semantic_layer.to_dict()
    if business_name not in sl:
        return {"error": f"'{business_name}' not in semantic layer",
                "available": sorted(sl.keys())}
    entry  = sl[business_name]
    col    = entry["column"]
    cols   = [c for c in ["Location", "ISO3", "Year", col] if c in dp.data.columns]
    df     = dp.data[cols].copy()
    if year is not None:
        df = df[df["Year"] == year]
    result = df.nlargest(top_n, col).reset_index(drop=True)
    return {"business_name": business_name, "resolved_column": col,
            "unit": entry["unit"], "year_filter": year, "top_n": top_n,
            "results": result.to_dict(orient="records")}


TOOL_FUNCTIONS_A = {
    "list_available_sources": list_available_sources,
    "load_source":            load_source,
    "merge_sources":          merge_sources,
    "query_product":          query_product,
}

TOOLS_ANTHROPIC_A = [
    {"name": "list_available_sources",
     "description": "Return available data sources and which are loaded.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "load_source",
     "description": "Load a source ('UN_WPP' or 'WORLD_BANK_GDP') and register it.",
     "input_schema": {"type": "object",
       "properties": {"source_name": {"type": "string"},
                      "year_start": {"type": "integer"},
                      "year_end": {"type": "integer"}},
       "required": ["source_name"]}},
    {"name": "merge_sources",
     "description": "Inner join two DataProducts on ISO3+Year.",
     "input_schema": {"type": "object",
       "properties": {"source1_name": {"type": "string"},
                      "source2_name": {"type": "string"},
                      "product_name": {"type": "string"}},
       "required": ["source1_name", "source2_name"]}},
    {"name": "query_product",
     "description": (
         "Query a merged product using a semantic business name. "
         "Available business names: 'net_migration_rate', 'net_migrants', "
         "'population', 'fertility_rate', 'life_expectancy', 'gdp', 'gdp_growth'. "
         "Returns top-N rows sorted by that metric."),
     "input_schema": {"type": "object",
       "properties": {"product_name": {"type": "string"},
                      "business_name": {"type": "string"},
                      "year": {"type": "integer"},
                      "top_n": {"type": "integer"}},
       "required": ["product_name", "business_name"]}},
]

TOOLS_OPENAI_A = [
    {"type": "function", "function": {
        "name": t["name"], "description": t["description"],
        "parameters": t["input_schema"]}}
    for t in TOOLS_ANTHROPIC_A
]

SYSTEM_PROMPT_A = (
    "You are a data analyst agent. Use the tools to answer the user's question.\n"
    "1. Call list_available_sources() to see what data is available.\n"
    "2. Call load_source() for each source you need.\n"
    "3. Call merge_sources() to combine them.\n"
    "4. Call query_product() with a business name to retrieve data.\n"
    "   Available business names: net_migration_rate, gdp, population, etc.\n"
    "Be concise. Report the country name and value in your final answer."
)

print(f"Condition A: {len(TOOL_FUNCTIONS_A)} tools (semantic layer active).")

Condition A: 4 tools (semantic layer active).


---
## Condition B — Without Semantic Layer

Agents use `list_columns()` to discover raw column names and
`query_raw(column_name=...)` to retrieve data.
No business names. No semantic mediation.

In [6]:
# ── Condition B: raw query tools (no semantic layer) ────────────────────────

def list_columns(product_name):
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    dp = PRODUCT_REGISTRY[product_name]
    return {"product_name": product_name,
            "columns": list(dp.data.columns),
            "shape": list(dp.data.shape)}


def query_raw(product_name, column_name, year=None, top_n=10):
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    dp = PRODUCT_REGISTRY[product_name]
    if column_name not in dp.data.columns:
        return {"error": f"'{column_name}' not a column in '{product_name}'",
                "available_columns": list(dp.data.columns)}
    cols   = [c for c in ["Location", "ISO3", "Year", column_name] if c in dp.data.columns]
    df     = dp.data[cols].copy()
    if year is not None:
        df = df[df["Year"] == year]
    result = df.nlargest(top_n, column_name).reset_index(drop=True)
    return {"column_name": column_name, "year_filter": year, "top_n": top_n,
            "results": result.to_dict(orient="records")}


TOOL_FUNCTIONS_B = {
    "list_available_sources": list_available_sources,
    "load_source":            load_source,
    "merge_sources":          merge_sources,
    "list_columns":           list_columns,
    "query_raw":              query_raw,
}

TOOLS_ANTHROPIC_B = [
    {"name": "list_available_sources",
     "description": "Return available data sources and which are loaded.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "load_source",
     "description": "Load a source ('UN_WPP' or 'WORLD_BANK_GDP') and register it.",
     "input_schema": {"type": "object",
       "properties": {"source_name": {"type": "string"},
                      "year_start": {"type": "integer"},
                      "year_end": {"type": "integer"}},
       "required": ["source_name"]}},
    {"name": "merge_sources",
     "description": "Inner join two DataProducts on ISO3+Year.",
     "input_schema": {"type": "object",
       "properties": {"source1_name": {"type": "string"},
                      "source2_name": {"type": "string"},
                      "product_name": {"type": "string"}},
       "required": ["source1_name", "source2_name"]}},
    {"name": "list_columns",
     "description": "List all raw column names available in a registered product.",
     "input_schema": {"type": "object",
       "properties": {"product_name": {"type": "string"}},
       "required": ["product_name"]}},
    {"name": "query_raw",
     "description": "Query a product by raw column name. Use list_columns() first to discover available columns. Returns top-N rows sorted by that column.",
     "input_schema": {"type": "object",
       "properties": {"product_name": {"type": "string"},
                      "column_name": {"type": "string"},
                      "year": {"type": "integer"},
                      "top_n": {"type": "integer"}},
       "required": ["product_name", "column_name"]}},
]

TOOLS_OPENAI_B = [
    {"type": "function", "function": {
        "name": t["name"], "description": t["description"],
        "parameters": t["input_schema"]}}
    for t in TOOLS_ANTHROPIC_B
]

SYSTEM_PROMPT_B = (
    "You are a data analyst agent. Use the tools to answer the user's question.\n"
    "1. Call list_available_sources() to see what data is available.\n"
    "2. Call load_source() for each source you need.\n"
    "3. Call merge_sources() to combine them.\n"
    "4. Call list_columns() to discover available column names.\n"
    "5. Call query_raw() with the appropriate column name to retrieve data.\n"
    "Be concise. Report the country name and value in your final answer."
)

print(f"Condition B: {len(TOOL_FUNCTIONS_B)} tools (no semantic layer — raw column names).")

Condition B: 5 tools (no semantic layer — raw column names).


In [7]:
MAX_ITERATIONS = 20


def run_agent(question, provider, model, condition_label,
              tools_anthropic, tools_openai, tool_functions,
              system_prompt, verbose=True):
    # Provider-agnostic agent loop parameterised by tool set.
    # condition_label: 'A' or 'B' for logging.
    PRODUCT_REGISTRY.clear()
    tool_calls_log = []

    if provider == "anthropic":
        messages = [{"role": "user", "content": question}]
    else:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question},
        ]

    for iteration in range(MAX_ITERATIONS):

        if provider == "anthropic":
            response = anthropic_client.messages.create(
                model=model, max_tokens=2048,
                system=system_prompt,
                tools=tools_anthropic,
                messages=messages)

            if response.stop_reason == "end_turn":
                text = next(
                    (b.text for b in response.content if hasattr(b, "text")), "")
                return {"answer": text, "tool_calls": tool_calls_log,
                        "iterations": iteration + 1,
                        "model": model, "provider": provider,
                        "condition": condition_label}

            if response.stop_reason == "tool_use":
                messages.append({"role": "assistant", "content": response.content})
                results = []
                for block in response.content:
                    if block.type == "tool_use":
                        tool_calls_log.append({"name": block.name, "input": block.input})
                        if verbose:
                            print(f"  -> {block.name}({block.input})")
                        fn     = tool_functions.get(block.name)
                        result = fn(**block.input) if fn else {"error": "unknown tool"}
                        results.append({"type": "tool_result",
                                        "tool_use_id": block.id,
                                        "content": json.dumps(result)})
                messages.append({"role": "user", "content": results})

        elif provider == "openai":
            response = openai_client.chat.completions.create(
                model=model, max_tokens=2048,
                tools=tools_openai,
                messages=messages)

            choice = response.choices[0]

            if choice.finish_reason == "stop":
                return {"answer": choice.message.content,
                        "tool_calls": tool_calls_log,
                        "iterations": iteration + 1,
                        "model": model, "provider": provider,
                        "condition": condition_label}

            if choice.finish_reason == "tool_calls":
                messages.append(choice.message)
                for tc in choice.message.tool_calls:
                    args = json.loads(tc.function.arguments)
                    tool_calls_log.append({"name": tc.function.name, "input": args})
                    if verbose:
                        print(f"  -> {tc.function.name}({args})")
                    fn     = tool_functions.get(tc.function.name)
                    result = fn(**args) if fn else {"error": "unknown tool"}
                    messages.append({"role": "tool",
                                     "tool_call_id": tc.id,
                                     "content": json.dumps(result)})

    return {"answer": "Max iterations reached.",
            "tool_calls": tool_calls_log, "iterations": MAX_ITERATIONS,
            "model": model, "provider": provider, "condition": condition_label}


print("run_agent() ready — parameterised by tool set and condition.")

run_agent() ready — parameterised by tool set and condition.


In [8]:
QUESTION = (
    "Which country in the top 20 GDP has the highest migration rate?"
)

MODELS = [
    {"provider": "openai",    "model": "gpt-4o-mini",               "label": "OpenAI GPT-4o-mini"},
    {"provider": "anthropic", "model": "claude-haiku-4-5-20251001",  "label": "Anthropic Haiku"},
    {"provider": "anthropic", "model": "claude-sonnet-4-6",          "label": "Anthropic Sonnet"},
]

print(f"Question : {QUESTION}")
print(f"Models   : {[m['label'] for m in MODELS]}")

Question : Which country in the top 20 GDP has the highest migration rate?
Models   : ['OpenAI GPT-4o-mini', 'Anthropic Haiku', 'Anthropic Sonnet']


---
## Run — Condition A (With Semantic Layer)

In [9]:
results_a = []

for cfg in MODELS:
    print(f"\n{'='*60}")
    print(f"[A] {cfg['label']}")
    print(f"{'='*60}")
    result = run_agent(
        QUESTION, cfg["provider"], cfg["model"],
        condition_label="A",
        tools_anthropic=TOOLS_ANTHROPIC_A,
        tools_openai=TOOLS_OPENAI_A,
        tool_functions=TOOL_FUNCTIONS_A,
        system_prompt=SYSTEM_PROMPT_A,
        verbose=True)
    result["label"] = cfg["label"]
    results_a.append(result)
    print(f"\nAnswer:\n{result['answer']}")


[A] OpenAI GPT-4o-mini
  -> list_available_sources({})
  -> load_source({'source_name': 'WORLD_BANK_GDP'})
  -> load_source({'source_name': 'UN_WPP'})
  -> query_product({'product_name': 'WORLD_BANK_GDP', 'business_name': 'gdp', 'top_n': 20})
  -> query_product({'product_name': 'UN_WPP', 'business_name': 'net_migration_rate'})
  -> query_product({'product_name': 'WORLD_BANK_GDP', 'business_name': 'gdp', 'top_n': 100})
  -> query_product({'product_name': 'UN_WPP', 'business_name': 'net_migration_rate', 'top_n': 100})

Answer:
The country in the top 20 GDP with the highest migration rate is **Kuwait** with a net migration rate of **340.853 per 1,000 population** (from the year 1991).

[A] Anthropic Haiku
  -> list_available_sources({})
  -> load_source({'source_name': 'UN_WPP'})
  -> load_source({'source_name': 'WORLD_BANK_GDP'})
  -> merge_sources({'source1_name': 'UN_WPP', 'source2_name': 'WORLD_BANK_GDP', 'product_name': 'combined_data'})
  -> query_product({'product_name': 'combined

---
## Run — Condition B (Without Semantic Layer)

In [10]:
results_b = []

for cfg in MODELS:
    print(f"\n{'='*60}")
    print(f"[B] {cfg['label']}")
    print(f"{'='*60}")
    result = run_agent(
        QUESTION, cfg["provider"], cfg["model"],
        condition_label="B",
        tools_anthropic=TOOLS_ANTHROPIC_B,
        tools_openai=TOOLS_OPENAI_B,
        tool_functions=TOOL_FUNCTIONS_B,
        system_prompt=SYSTEM_PROMPT_B,
        verbose=True)
    result["label"] = cfg["label"]
    results_b.append(result)
    print(f"\nAnswer:\n{result['answer']}")


[B] OpenAI GPT-4o-mini
  -> list_available_sources({})
  -> load_source({'source_name': 'WORLD_BANK_GDP'})
  -> load_source({'source_name': 'UN_WPP'})
  -> list_columns({'product_name': 'WORLD_BANK_GDP'})
  -> list_columns({'product_name': 'UN_WPP'})
  -> query_raw({'product_name': 'WORLD_BANK_GDP', 'column_name': 'GDP_USD', 'top_n': 20})
  -> query_raw({'product_name': 'UN_WPP', 'column_name': 'NetMigrationRate_per_Kpop'})
  -> query_raw({'product_name': 'UN_WPP', 'column_name': 'NetMigrationRate_per_Kpop', 'year': 2023, 'top_n': 20})

Answer:
The country with the highest migration rate among the top 20 GDP countries is the **United Arab Emirates**, with a migration rate of **28.19** per 1,000 population.

[B] Anthropic Haiku
  -> list_available_sources({})
  -> load_source({'source_name': 'WORLD_BANK_GDP'})
  -> load_source({'source_name': 'UN_WPP'})
  -> merge_sources({'source1_name': 'WORLD_BANK_GDP', 'source2_name': 'UN_WPP', 'product_name': 'merged_gdp_migration'})
  -> query_ra

---
## Results

In [11]:
def summarise(results, condition_label):
    rows = []
    for r in results:
        correct = 'canada' in r['answer'].lower()
        # For condition A: look for query_product calls
        # For condition B: look for query_raw calls and note column chosen
        if condition_label == "A":
            query_calls = [tc for tc in r['tool_calls']
                          if tc['name'] == 'query_product']
            key_col = [tc['input'].get('business_name') for tc in query_calls]
        else:
            query_calls = [tc for tc in r['tool_calls']
                          if tc['name'] == 'query_raw']
            key_col = [tc['input'].get('column_name') for tc in query_calls]
        year_filters = [tc['input'].get('year') for tc in query_calls
                       if tc['input'].get('year')]
        rows.append({
            'Condition': condition_label,
            'Model': r['label'],
            'Tool calls': len(r['tool_calls']),
            'Columns queried': str(key_col),
            'Year filter': str(year_filters) if year_filters else 'none',
            'Canada correct?': 'YES' if correct else 'NO',
        })
    return rows

rows = summarise(results_a, 'A') + summarise(results_b, 'B')
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

Condition              Model  Tool calls                                                                  Columns queried  Year filter Canada correct?
        A OpenAI GPT-4o-mini           7                       ['gdp', 'net_migration_rate', 'gdp', 'net_migration_rate']         none              NO
        A    Anthropic Haiku           8                       ['gdp', 'net_migration_rate', 'gdp', 'net_migration_rate'] [2023, 2023]             YES
        A   Anthropic Sonnet           8                       ['gdp', 'net_migration_rate', 'gdp', 'net_migration_rate'] [2023, 2023]             YES
        B OpenAI GPT-4o-mini           8            ['GDP_USD', 'NetMigrationRate_per_Kpop', 'NetMigrationRate_per_Kpop']       [2023]              NO
        B    Anthropic Haiku           7                                  ['gdp', 'GDP_USD', 'NetMigrationRate_per_Kpop']       [2023]             YES
        B   Anthropic Sonnet           9 ['GDP_USD', 'NetMigrationRate_per_Kpop', 'NetMigratio

---
## Findings

**Verified answer:** Canada, +11.039 per 1,000 population (2023, UN WPP)

---

### Results summary — Run 1 (2026-06-30)

| | Condition A — With Semantic Layer | Condition B — Without Semantic Layer |
|---|---|---|
| **GPT-4o-mini** | ❌ China, 357.14 (1964) | ❌ UAE, 128.43 (2007) |
| **Haiku** | ✅ Canada, 11.039 (2023) | ✅ Canada, 11.039 (2023) |
| **Sonnet** | ❌ Singapore, 41.83 (2007) | ❌ Singapore (same failure) |

### Results summary — Run 2 (2026-06-30)

| | Condition A — With Semantic Layer | Condition B — Without Semantic Layer |
|---|---|---|
| **GPT-4o-mini** | ❌ Kuwait, 340.85 (1991) | ❌ UAE, 28.19 (2023) |
| **Haiku** | ✅ Canada, 11.039 (2023) | ✅ Canada, 11.039 (2023) |
| **Sonnet** | ✅ Canada, 11.039 (2023) | ✅ Canada, 11.039 (2023) |

### Cumulative correctness (2 runs × 2 conditions = 4 observations per model)

| Model | Correct | Total | Rate |
|---|---|---|---|
| GPT-4o-mini | 0 | 4 | **0/4** |
| Haiku | 4 | 4 | **4/4** |
| Sonnet | 2 | 4 | **2/4** |

---

### Behavioral profiles — Condition A (With Semantic Layer)

**GPT-4o-mini — WRONG both runs**

| | Run 1 | Run 2 |
|---|---|---|
| Tool calls | 6 | 7 |
| Merge step | ✅ merge_sources() called | ❌ skipped — queried individual products |
| GDP query | `merged` product, top_n=20 | `WORLD_BANK_GDP` directly, top_n=20 then 100 |
| Migration query | `merged` product, top_n=20 | `UN_WPP` directly, no year filter |
| Year filter | None | None |
| Answer | China, 357.14 (1964) | Kuwait, 340.85 (1991) |

Run 2 introduced a new failure mode: GPT-4o-mini skipped `merge_sources()` entirely and queried the two source products separately, then attempted cross-dataset reasoning in its own response. Different wrong answer, same root cause: no governed cross-metric join and no year filter.

**Haiku — CORRECT both runs**

| | Run 1 | Run 2 |
|---|---|---|
| Tool calls | 8 | 8 |
| Load order | WB_GDP → UN_WPP | UN_WPP → WB_GDP |
| Merge step | ✅ | ✅ |
| Round 1 | No year filter | No year filter |
| Round 2 | year=2023 applied | year=2023 applied |
| Answer | ✅ Canada, 11.039 | ✅ Canada, 11.039 |

Haiku's two-round strategy (broad first, then year-filtered) was identical across both runs.
Load order flipped between runs but correctness was unaffected.

**Sonnet — WRONG Run 1, CORRECT Run 2**

| | Run 1 | Run 2 |
|---|---|---|
| Tool calls | 6 | 8 |
| Merge step | ✅ | ✅ |
| Round 1 | top_n=200, no year | top_n=20, no year |
| Round 2 | — (stopped) | top_n=20, year=2023 ✅ |
| Year filter | Never | Applied in Round 2 |
| Answer | ❌ Singapore, 41.83 (2007) | ✅ Canada, 11.039 (2023) |

Sonnet's year filter decision is probabilistic. Run 1: stopped after one wide query.
Run 2: added a second round with year=2023 — same strategy as Haiku. Two extra tool calls, correct result.

---

### Behavioral profiles — Condition B (Without Semantic Layer)

**GPT-4o-mini — WRONG both runs**

| | Run 1 | Run 2 |
|---|---|---|
| Tool calls | 10 | 8 |
| Merge step | ✅ merged first | ❌ skipped — queried individual products |
| Semantic leakage | ✅ tried 'gdp', 'net_migration_rate' first | ❌ none — list_columns() first |
| Column discovery | Error-guided recovery | list_columns() on each source |
| Year filter | None | year=2023 on migration only |
| Answer | UAE, 128.43 (2007) | UAE, 28.19 (2023) |

Run 2: no semantic leakage — immediately called `list_columns()` and chose correct column names.
Applied `year=2023` to the migration query but still queried unmerged sources.
UAE appeared at top of 2023 migration in UN_WPP, but UAE is not in the top-20 GDP list —
the cross-dataset comparison failed because no merge was performed.

**Haiku — CORRECT both runs**

| | Run 1 | Run 2 |
|---|---|---|
| Tool calls | 9 | 7 |
| Merge step | ✅ | ✅ |
| Semantic leakage | ✅ tried 'gdp' first | ✅ tried 'gdp' first |
| Recovery | Error message → GDP_USD | Error message → GDP_USD |
| Year filter | year=2023 Round 2 | year=2023 applied directly |
| Answer | ✅ Canada, 11.039 | ✅ Canada, 11.039 |

Haiku showed semantic leakage in both runs — tried 'gdp' as a raw column name each time.
Run 2 was more efficient (7 vs 9 calls) — applied year filter directly rather than in a second round.

**Sonnet — WRONG Run 1, CORRECT Run 2**

| | Run 1 | Run 2 |
|---|---|---|
| Tool calls | 7 | 9 |
| Merge step | ✅ | ✅ |
| Semantic leakage | ❌ none | ❌ none |
| Column discovery | list_columns() → correct first try | list_columns() → correct first try |
| Round 1 | top_n=200, no year | top_n=200, no year |
| Round 2 | — (stopped) | top_n=500, year=2023 ✅ + GDP year=2023 |
| Year filter | Never | Applied in Round 2 |
| Answer | ❌ Singapore, 41.83 (2007) | ✅ Canada, 11.039 (2023) |

Sonnet never shows semantic leakage in Condition B — calls `list_columns()` and picks correctly.
Year filter decision is the same probabilistic variable as in Condition A.

---

### The semantic leakage finding (updated)

Haiku showed semantic leakage in both runs of Condition B.
GPT-4o-mini showed leakage in Run 1 but not Run 2 — the prior is probabilistic.
Sonnet never shows leakage — uses `list_columns()` as intended.

| Model | Run 1 leakage | Run 2 leakage | Pattern |
|---|---|---|---|
| GPT-4o-mini | ✅ | ❌ | Probabilistic |
| Haiku | ✅ | ✅ | Consistent |
| Sonnet | ❌ | ❌ | Never |

---

### The discovery tax (updated across runs)

| Model | Cond A avg calls | Cond B avg calls | Extra calls |
|---|---|---|---|
| GPT-4o-mini | 6.5 | 9.0 | +2.5 |
| Haiku | 8.0 | 8.0 | 0 |
| Sonnet | 7.0 | 8.0 | +1.0 |

Haiku's discovery tax dropped to zero in Run 2 — it recovered from the failed column guess
efficiently enough that total call count matched Condition A.

---

### The core failure mode: year filter and merge step

Two independent failure modes observed:

**1. No year filter:** Without anchoring to 2023, the migration query returns historical peaks.
China 1964 (357.14), Kuwait 1991 (340.85), UAE 2007 (128.43), Singapore 2007 (41.83)
all dominate when all years are considered.

**2. Skipped merge (GPT-4o-mini, Run 2):** Querying individual source products without merging
means the cross-metric comparison happens in LLM reasoning across two disconnected result sets.
This fails when UAE appears in the top migration list but is not in the GDP list the model saw.

| Failure type | Run 1 | Run 2 | Probabilistic? |
|---|---|---|---|
| No year filter | GPT-4o-mini, Sonnet | GPT-4o-mini | Yes — Sonnet corrected itself |
| Skipped merge | — | GPT-4o-mini (both conditions) | Yes — new failure mode |
| Wrong column name | GPT-4o-mini, Haiku (Cond B) | Haiku (Cond B) | Yes — GPT-4o-mini corrected |

---

### Hypothesis verdict (updated after 2 runs)

> **"LLM agents reason more accurately and consistently when querying data
> through semantic business names than when querying raw technical column names."**

| Claim | 2-run verdict |
|---|---|
| **Correctness** | Same across conditions — naming did not determine correctness in either run. |
| **Efficiency** | Confirmed in Run 1. Reduced in Run 2 — Haiku closed the gap completely. |
| **Alignment** | Partially confirmed — Haiku shows consistent semantic leakage; GPT-4o-mini probabilistic; Sonnet never. |
| **Failure isolation** | Confirmed — semantic layer is irrelevant to year filter and merge-skip failures. |

**Verdict: Necessary but not sufficient — and the gap is smaller than Run 1 suggested.**

The discovery tax is real but variable. Semantic leakage is model-dependent and probabilistic.
The decisive failure modes — year filter and skipped merge — are independent of the semantic layer
and account for all wrong answers across both runs.

---

### Connection to neurosymbolic AI

The symbolic semantic layer provides:
- Governed column resolution (naming)
- Alignment to model vocabulary (leakage prevention)

It cannot provide:
- Temporal anchoring (year filter) — probabilistic reasoning choice
- Merge enforcement — no tool *requires* merge before query
- Cross-metric filtering — the structural gap from Experiment 1 remains open

**Symbolic scaffolding reduces the space where neural failures can occur,
but does not eliminate the probabilistic component.**

---

### Next experiment

**Experiment 3 candidate — Close the structural gap.**

Add a `query_product_filtered_by(metric, filter_metric, filter_top_n, year)` tool that
performs the cross-metric join in one governed call, with year required. Test whether
collapsing the two-step reasoning into a single structured tool eliminates divergence.

If it does, the failure was structural (missing tool).
If it does not, the failure is neural (reasoning quality independent of tool design).

> See `machine_learning/agentic_model_reasoning_divergence.ipynb` (Experiment 1)
> and `machine_learning/agentic_semantic_layer_necessity_slides.md` (these slides)

---
## References

- **Data:** UN WPP 2024 (population.un.org/wpp) · World Bank WDI (databank.worldbank.org)
- **Anthropic Tool Use:** docs.anthropic.com/en/docs/tool-use
- **OpenAI Function Calling:** platform.openai.com/docs/guides/function-calling
- **Data products:** Dehghani (2022) *Data Mesh*, O'Reilly
- **Prior experiment:** `machine_learning/agentic_model_reasoning_divergence.ipynb`
- **Support module:** `../python_vignettes/data_products/data_product_lib.py`